# All-in-one real-world multimodal demo

This notebook is the recommended path for recording the demo. It consolidates the earlier step notebooks into one linear Colab flow.

It demonstrates:

1. project motivation and conceptual architecture;
2. real yfinance OHLCV data download;
3. feature and label creation;
4. aligned multimodal artifact creation;
5. fusion ablations;
6. actual-data visuals for the recording.

**Important:** this notebook creates real-data embeddings and ablation metrics. It does not implement or claim a full portfolio backtest.

## 1. Conceptual framing

Traditional stock prediction often starts with price and volume features. But a real analyst looks at multiple views at once: numbers, chart structure, text context, and peer/sector/event relationships.

This project asks whether a Transformer can combine those views into one aligned representation keyed by:

```text
stock_id + end_date
```

In [ ]:
# Conceptual diagram: raw multimodal inputs -> aligned embedding artifact.
# This is explanatory only, not a result/backtest chart.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def box(ax, xy, text, width=2.6, height=0.72, fc='#eef6ff', ec='#4a90e2', fontsize=10):
    patch = FancyBboxPatch(
        xy, width, height,
        boxstyle='round,pad=0.05,rounding_size=0.08',
        linewidth=1.4, edgecolor=ec, facecolor=fc
    )
    ax.add_patch(patch)
    ax.text(xy[0] + width / 2, xy[1] + height / 2, text, ha='center', va='center', fontsize=fontsize)

def arrow(ax, start, end):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle='->', mutation_scale=14, linewidth=1.2, color='#555555'))

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis('off')

ax.text(0.4, 6.5, 'Raw multimodal inputs', fontsize=14, weight='bold')
box(ax, (0.4, 5.35), 'Tabular market data\nOHLCV, returns, volatility')
box(ax, (0.4, 4.25), 'Chart images\nCandlestick PNG windows', fc='#f5efff', ec='#8e5cc2')
box(ax, (0.4, 3.15), 'Text records\nevent_date <= sample date', fc='#eefaf1', ec='#4c9f70')
box(ax, (0.4, 2.05), 'KG context\nsector, peer, event graph', fc='#fff5e8', ec='#d68a2d')

ax.text(4.0, 6.5, 'Preprocess and align', fontsize=14, weight='bold')
box(ax, (4.0, 5.0), 'Feature engineering\nfuture outperformance label', width=3.0)
box(ax, (4.0, 3.9), 'Chart generation\nas-of-date window', width=3.0, fc='#f5efff', ec='#8e5cc2')
box(ax, (4.0, 2.8), 'Text/KG cutoff\nno future leakage', width=3.0, fc='#eefaf1', ec='#4c9f70')
box(ax, (4.0, 1.7), 'Alignment key\nstock_id + end_date', width=3.0, fc='#fff5e8', ec='#d68a2d')

ax.text(8.2, 6.5, 'Embedding artifact', fontsize=14, weight='bold')
box(ax, (8.25, 4.95), 'tabular_tokens', width=2.8)
box(ax, (8.25, 4.05), 'image_tokens', width=2.8, fc='#f5efff', ec='#8e5cc2')
box(ax, (8.25, 3.15), 'text_tokens', width=2.8, fc='#eefaf1', ec='#4c9f70')
box(ax, (8.25, 2.25), 'kg_tokens', width=2.8, fc='#fff5e8', ec='#d68a2d')
box(ax, (8.25, 1.25), 'y + metadata', width=2.8, fc='#f2f2f2', ec='#777777')

for y in [5.7, 4.6, 3.5, 2.4]:
    arrow(ax, (3.05, y), (3.95, y - 0.2))
for y in [5.35, 4.25, 3.15, 2.05]:
    arrow(ax, (7.05, y - 0.05), (8.2, y - 0.05))

ax.text(0.4, 0.45, 'Takeaway: heterogeneous raw data becomes synchronized vectors for the same stock/date sample.', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Conceptual diagram: modality embeddings -> fusion Transformer -> prediction.
# This is architecture framing only, not a result chart.
fig, ax = plt.subplots(figsize=(14, 6.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6.5)
ax.axis('off')

ax.text(0.4, 6.0, 'Aligned modality embeddings', fontsize=14, weight='bold')
box(ax, (0.5, 4.9), 'tabular embedding', width=2.4)
box(ax, (0.5, 3.9), 'image embedding', width=2.4, fc='#f5efff', ec='#8e5cc2')
box(ax, (0.5, 2.9), 'text embedding', width=2.4, fc='#eefaf1', ec='#4c9f70')
box(ax, (0.5, 1.9), 'kg embedding', width=2.4, fc='#fff5e8', ec='#d68a2d')

ax.text(4.35, 6.0, 'Fusion in shared embedding space', fontsize=14, weight='bold')
box(ax, (4.1, 4.75), 'Project to common dimension', width=3.2, fc='#eef6ff')
box(ax, (4.1, 3.75), 'Concatenate modality tokens', width=3.2, fc='#eef6ff')
box(ax, (4.1, 2.75), 'Self-attention across modalities', width=3.2, fc='#eef6ff')
box(ax, (4.1, 1.75), 'Fused representation', width=3.2, fc='#eef6ff')

ax.text(8.65, 6.0, 'Outputs and interpretation', fontsize=14, weight='bold')
box(ax, (8.7, 4.75), 'Probability of\noutperformance vs Nifty', width=2.8, fc='#f2f2f2', ec='#777777')
box(ax, (8.7, 3.45), 'Cross-modal reasoning\ntrend + chart + text + KG', width=2.8, fc='#f2f2f2', ec='#777777')
box(ax, (8.7, 2.15), 'Ablation comparison\nwhich modality changes behavior?', width=2.8, fc='#f2f2f2', ec='#777777')

for y in [5.25, 4.25, 3.25, 2.25]:
    arrow(ax, (2.95, y), (4.05, 4.95))
arrow(ax, (7.35, 3.1), (8.65, 5.1))
arrow(ax, (7.35, 3.1), (8.65, 3.8))
arrow(ax, (7.35, 3.1), (8.65, 2.5))

ax.text(0.5, 0.55, 'Takeaway: fusion lets the model mix signals that a single modality would see in isolation.', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

## 2. Setup Colab and project dependencies

Run the cells below top-to-bottom. If you already cloned the repo in this runtime, the clone step will be skipped.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Randhir123/nifty50-multimodal-transformer.git'
REPO_DIR = '/content/nifty50-multimodal-transformer'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Installed project dependencies.')

## 3. Demo configuration

Start with the default three-stock universe. Once the full notebook works, you can add more tickers for richer visuals.

In [ ]:
from datetime import date, timedelta

TICKERS = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS']
BENCHMARK = '^NSEI'
SECTORS = {'RELIANCE.NS': 'Energy', 'TCS.NS': 'IT', 'INFY.NS': 'IT'}

END = date.today().isoformat()
START = (date.today() - timedelta(days=31 * 9)).isoformat()
OUTPUT_DIR = Path('data/processed/real_world_demo')
RAW_DIR = OUTPUT_DIR / 'raw'
CHART_DIR = OUTPUT_DIR / 'charts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 20
HORIZON_DAYS = 3
CHART_LOOKBACK_DAYS = 20

FEATURE_COLUMNS = [
    'log_return_1d',
    'cum_return_3d',
    'cum_return_5d',
    'cum_return_10d',
    'realized_vol_5d',
    'realized_vol_10d',
    'high_low_range_over_close',
    'close_over_10dma_minus_1',
    'close_over_20dma_minus_1',
    'volume_over_20d_avg',
    'stock_minus_index_return',
]

print('Tickers:', TICKERS)
print('Benchmark:', BENCHMARK)
print('Date range:', START, 'to', END)
print('Output:', OUTPUT_DIR)

## 4. Download real OHLCV data

This cell downloads stock and benchmark data from yfinance and caches it under `data/processed/real_world_demo/raw/`.

In [ ]:
import pandas as pd
import numpy as np

from src.data.download_yfinance import (
    download_multiple_tickers,
    download_benchmark_data,
    save_ticker_csv,
    deterministic_csv_path_for_ticker,
)

stock_data = download_multiple_tickers(TICKERS, start=START, end=END)
for ticker, df in stock_data.items():
    path = save_ticker_csv(ticker, df, output_dir=RAW_DIR)
    print(ticker, '->', path, df.shape)

benchmark_df = download_benchmark_data(BENCHMARK, start=START, end=END)
benchmark_path = save_ticker_csv(BENCHMARK, benchmark_df, output_dir=RAW_DIR)
print(BENCHMARK, '->', benchmark_path, benchmark_df.shape)

display(stock_data[TICKERS[0]].head())
display(benchmark_df.head())

## 5. Build features and labels

The label is `1` when the stock outperforms the Nifty benchmark over the future horizon, otherwise `0`.

In [ ]:
from src.data.features import compute_technical_features
from src.data.labels import generate_outperformance_label

frames = []
for ticker in TICKERS:
    features = compute_technical_features(stock_data[ticker], benchmark_df)
    labelled = generate_outperformance_label(features, horizon_days=HORIZON_DAYS)
    labelled['stock_id'] = ticker
    frames.append(labelled)

combined = pd.concat(frames, ignore_index=True)
required = ['stock_id', 'date', 'label', *FEATURE_COLUMNS]
tabular_df = combined.dropna(subset=required).loc[:, required + ['close', 'volume']].reset_index(drop=True)
tabular_csv = OUTPUT_DIR / 'tabular_samples.csv'
tabular_df.to_csv(tabular_csv, index=False)

print('Wrote:', tabular_csv)
print('Rows:', len(tabular_df))
display(tabular_df.head())
display(tabular_df.groupby('stock_id')['label'].agg(['count', 'sum', 'mean']).rename(columns={'sum': 'positive_labels', 'mean': 'positive_rate'}))

## 6. Build aligned multimodal tensors

This section creates the main artifact: `real_world_multimodal_samples.npz`. Every modality is aligned row-by-row using `stock_id + end_date`.

In [ ]:
from src.data.multimodal_samples import (
    attach_image_tokens,
    attach_kg_tokens,
    attach_text_tokens,
    build_tabular_multimodal_samples,
    save_multimodal_samples,
)
from src.kg.build_graph import build_market_knowledge_graph
from src.models.image_transformer import ImageTransformerConfig
from src.viz.charts import generate_or_resolve_sample_chart

arrays = build_tabular_multimodal_samples(
    tabular_df,
    feature_cols=FEATURE_COLUMNS,
    window_size=WINDOW_SIZE,
)
print('Tabular tokens:', arrays.tabular_tokens.shape)
print('Samples:', len(arrays.y))

In [ ]:
# Text records: market-summary text generated from real OHLCV-derived features.
records = []
for ticker, frame in tabular_df.groupby('stock_id'):
    frame = frame.sort_values('date').reset_index(drop=True)
    for idx in range(0, len(frame), 5):
        row = frame.iloc[idx]
        direction = 'positive' if row['log_return_1d'] >= 0 else 'negative'
        records.append({
            'stock_id': ticker,
            'event_date': row['date'],
            'source_type': 'market_summary',
            'title': f'{ticker} {direction} daily market summary',
            'body_text': (
                f"As of {row['date'].date()}, {ticker} had a {direction} one-day return "
                f"of {row['log_return_1d']:.4f}, relative return versus index "
                f"of {row['stock_minus_index_return']:.4f}, and volume ratio "
                f"{row['volume_over_20d_avg']:.4f}."
            ),
        })

text_records = pd.DataFrame(records)
text_records.to_csv(OUTPUT_DIR / 'text_records.csv', index=False)
display(text_records.head())

In [ ]:
# KG inputs: sector mapping, recent returns, and high-volume event flags.
stock_sectors = pd.DataFrame([{'stock_id': t, 'sector_id': SECTORS.get(t, 'Unknown')} for t in TICKERS])
stock_sectors.to_csv(OUTPUT_DIR / 'stock_sectors.csv', index=False)

kg_returns = tabular_df.loc[:, ['stock_id', 'date', 'stock_minus_index_return']].rename(
    columns={'stock_minus_index_return': 'recent_return'}
)
kg_returns.to_csv(OUTPUT_DIR / 'kg_returns.csv', index=False)

events = []
for ticker, frame in tabular_df.groupby('stock_id'):
    top_volume = frame['volume_over_20d_avg'].quantile(0.9)
    for _, row in frame.loc[frame['volume_over_20d_avg'] >= top_volume].iterrows():
        events.append({'stock_id': ticker, 'event_date': row['date'], 'event_type': 'high_volume'})
event_records = pd.DataFrame(events) if events else pd.DataFrame(columns=['stock_id', 'event_date', 'event_type'])
event_records.to_csv(OUTPUT_DIR / 'event_records.csv', index=False)

display(stock_sectors)
display(event_records.head())

In [ ]:
# Generate real candlestick chart PNGs for each aligned sample.
for stock_id, end_date in zip(arrays.stock_ids, arrays.end_dates):
    generate_or_resolve_sample_chart(
        stock_data[str(stock_id)],
        symbol=str(stock_id),
        prediction_date=pd.Timestamp(end_date),
        output_dir=CHART_DIR,
        lookback_days=CHART_LOOKBACK_DAYS,
    )

print('Charts generated:', len(list(CHART_DIR.glob('*.png'))))

In [ ]:
# Attach text, KG, and image tokens.
graph = build_market_knowledge_graph(SECTORS, event_records=event_records)
arrays = attach_text_tokens(arrays, text_records, dim=16)
arrays = attach_kg_tokens(arrays, graph, returns=kg_returns)
arrays = attach_image_tokens(
    arrays,
    chart_dir=CHART_DIR,
    config=ImageTransformerConfig(
        image_size=64,
        patch_size=16,
        model_dim=16,
        num_heads=4,
        num_layers=1,
        ff_dim=32,
    ),
    device='cpu',
)

dataset_path = save_multimodal_samples(arrays, OUTPUT_DIR / 'real_world_multimodal_samples.npz')
print('Wrote:', dataset_path)

data = np.load(dataset_path, allow_pickle=False)
for key in data.files:
    print(f'{key:16s}', data[key].shape, data[key].dtype)

In [ ]:
# Display a few real chart images.
from IPython.display import Image, display
for path in list(CHART_DIR.glob('*.png'))[:3]:
    print(path.name)
    display(Image(filename=str(path)))

## 7. Run fusion ablations

This compares modality combinations on the same aligned artifact. These are classification/ablation metrics, not a portfolio backtest.

In [ ]:
cmd = [
    sys.executable,
    'scripts/run_ablation_study.py',
    '--dataset', str(dataset_path),
    '--output-dir', str(OUTPUT_DIR / 'ablations'),
    '--epochs', '1',
    '--batch-size', '4',
    '--device', 'cpu',
    '--model-dim', '16',
    '--num-heads', '4',
    '--num-layers', '1',
    '--ff-dim', '32',
    '--val-fraction', '0.25',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

results = pd.read_csv(OUTPUT_DIR / 'ablations' / 'ablation_results.csv')
display(results)

In [ ]:
# Plot ablation metrics from actual run outputs.
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
plot_df = results.set_index('variant')[metrics]
ax = plot_df.plot(kind='bar', figsize=(12, 5))
ax.set_title('Ablation metrics from real-world demo artifact')
ax.set_ylabel('Metric')
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
ablation_chart = OUTPUT_DIR / 'ablation_metrics_bar_chart.png'
plt.savefig(ablation_chart, dpi=160)
print('Wrote:', ablation_chart)
plt.show()

## 8. Visualize actual modality embeddings

This section projects actual generated modality vectors into 2D using PCA. These visuals come from the real demo artifact.

In [ ]:
from sklearn.decomposition import PCA

data = np.load(dataset_path, allow_pickle=False)
modalities = {
    'tabular': data['tabular_tokens'].mean(axis=1),
    'image': data['image_tokens'],
    'text': data['text_tokens'],
    'kg': data['kg_tokens'],
}

rows = []
for modality, values in modalities.items():
    n_components = 2 if values.shape[1] >= 2 else 1
    projection = PCA(n_components=n_components).fit_transform(values)
    if n_components == 1:
        projection = np.c_[projection[:, 0], np.zeros(values.shape[0])]
    for i, (x, y) in enumerate(projection):
        rows.append({
            'sample_id': i,
            'stock_id': str(data['stock_ids'][i]),
            'end_date': str(data['end_dates'][i]),
            'label': int(data['y'][i]),
            'modality': modality,
            'x': float(x),
            'y': float(y),
        })

projection_df = pd.DataFrame(rows)
projection_csv = OUTPUT_DIR / 'modality_embedding_projection.csv'
projection_df.to_csv(projection_csv, index=False)
print('Wrote:', projection_csv)
display(projection_df.head())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for modality, frame in projection_df.groupby('modality'):
    ax.scatter(frame['x'], frame['y'], label=modality, alpha=0.75)

ax.set_title('Actual modality embedding projections from real-world demo artifact')
ax.set_xlabel('PCA dimension 1')
ax.set_ylabel('PCA dimension 2')
ax.legend()
plt.tight_layout()
projection_png = OUTPUT_DIR / 'modality_embedding_projection.png'
plt.savefig(projection_png, dpi=160)
print('Wrote:', projection_png)
plt.show()

## 9. Write demo gallery and checklist

These markdown files are useful when recording the demo.

In [ ]:
gallery_path = OUTPUT_DIR / 'sample_gallery.md'
lines = ['# Real Demo Sample Gallery', '']
for i in range(min(10, len(data['stock_ids']))):
    stock_id = str(data['stock_ids'][i])
    end_date = pd.Timestamp(data['end_dates'][i]).strftime('%Y%m%d')
    label = int(data['y'][i])
    chart_name = f'{stock_id}_{end_date}.png'
    lines.extend([
        f'## Sample {i}: {stock_id} / {end_date}',
        '',
        f'- label: `{label}`',
        f'- chart: `charts/{chart_name}`',
        '',
        f'![{chart_name}](charts/{chart_name})',
        '',
    ])
gallery_path.write_text('\n'.join(lines), encoding='utf-8')

summary_path = OUTPUT_DIR / 'demo_visual_summary.md'
summary_lines = [
    '# Demo Visual Summary',
    '',
    'Generated from actual real-world demo artifacts.',
    '',
    '## Files',
    '',
    '- `real_world_multimodal_samples.npz`',
    '- `ablation_metrics_bar_chart.png`',
    '- `modality_embedding_projection.png`',
    '- `sample_gallery.md`',
    '',
    '## Recording message',
    '',
    'Raw multimodal market data has been converted into aligned tensors, fused by a Transformer, and evaluated through ablations. A full portfolio backtest remains future work.',
]
summary_path.write_text('\n'.join(summary_lines), encoding='utf-8')

print('Wrote:', gallery_path)
print('Wrote:', summary_path)
print('Demo output directory:', OUTPUT_DIR)

## Final checklist

You should now have:

```text
data/processed/real_world_demo/
├── raw/
├── charts/
├── tabular_samples.csv
├── text_records.csv
├── stock_sectors.csv
├── kg_returns.csv
├── event_records.csv
├── real_world_multimodal_samples.npz
├── ablations/ablation_results.csv
├── ablation_metrics_bar_chart.png
├── modality_embedding_projection.png
├── modality_embedding_projection.csv
├── sample_gallery.md
└── demo_visual_summary.md
```

Use these actual artifacts for the recording. Do not present ablation metrics as a full investment backtest.